# ScholarScan — NLP Preprocessing Pipeline

Cleans raw OCR text before similarity scoring.

## Pipeline
```
Raw OCR text
    │
    ├─► SBERT path  (semantic similarity)
    │       ↓
    │   sanitize → fix_split_words → domain_correct
    │       ↓
    │   Clean sentence text (preserves punctuation for SBERT)
    │
    └─► TF-IDF path  (lexical similarity)
            ↓
        sanitize → fix_split_words → domain_correct
            → tokenize → remove_stopwords → lemmatize
            ↓
        Bag-of-lemmas string
```

### What each step fixes
| Step | Problem | Example fix |
|------|---------|-------------|
| Sanitize | Unicode quotes, non-ASCII, OCR garbage | `\u2018hello\u2019` → `'hello'` |
| Fix split words | OCR breaks mid-word | `applica tion` → `application` |
| Domain correct | OCR misreads domain terms | `kubernates` → `kubernetes` |
| Tokenize | Split into words | sentence → token list |
| Remove stopwords | Filter common words | drop `is`, `the`, `and` |
| Lemmatize | Reduce to base form | `running` → `run` |

In [ ]:
!pip install -q spacy rapidfuzz scikit-learn
!python -m spacy download en_core_web_sm -q

In [ ]:
import re, logging
from functools import lru_cache
from rapidfuzz import fuzz, process
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s')
print('✓ Imports OK')

## Domain Vocabulary

Words the system recognises and protects from stopword removal / fuzzy correction.  
Currently configured for **cloud / Kubernetes** domain — extend for other subjects.

In [ ]:
# Source: backend/core/nlp.py — domain configuration

DOMAIN_VOCAB: dict[str, tuple[str, ...]] = {
    "cloud": (
        "application", "chart", "configuration", "crd", "deployment",
        "helm", "integration", "kubernetes", "postgresql", "redis",
        "serverless", "service", "template", "yaml",
    )
}

DOMAIN_HINTS: dict[str, frozenset[str]] = {
    "cloud": frozenset({"helm", "kubernetes", "yaml"})
}

_PROTECTED_COMMON_WORDS = frozenset(ENGLISH_STOP_WORDS).union({
    "answer", "class", "design", "effect", "explain", "method",
    "model", "paper", "process", "question", "result", "student",
    "system", "theory",
})

_SPLIT_SUFFIXES = frozenset({
    "able", "ation", "ality", "ative", "ence", "ency", "ible", "ical",
    "ified", "ifier", "iness", "ingly", "integration", "isation",
    "ization", "ition", "itive", "ively", "lement", "lessly", "logy",
    "ment", "ments", "ness", "ology", "sion", "tion",
})

_TOKEN_PATTERN = re.compile(r"\b\w+\b")
_TOKEN_OR_SEPARATOR_PATTERN = re.compile(r"\w+|\W+")
_FUZZY_SCORE_THRESHOLD = 80
_FUZZY_MARGIN_THRESHOLD = 5
_MIN_FUZZY_WORD_LENGTH = 4

@lru_cache(maxsize=1)
def _all_domain_terms() -> frozenset[str]:
    return frozenset(term for vocab in DOMAIN_VOCAB.values() for term in vocab)

@lru_cache(maxsize=1)
def _stop_words() -> frozenset[str]:
    return frozenset(ENGLISH_STOP_WORDS)

print('✓ Domain vocabulary configured')
print(f'  Terms: {sorted(_all_domain_terms())}')

## Step 1 — Sanitize

Normalises encoding and removes OCR garbage while preserving sentence structure.

In [ ]:
# Source: backend/core/nlp.py — _sanitize

_OCR_TRANSLATION_TABLE = str.maketrans({
    "\u2018": "'", "\u2019": "'",
    "\u201c": '"', "\u201d": '"',
    "\u2013": "-", "\u2014": "-",
    "\u00a0": " ",
})

def _sanitize(text: str) -> str:
    """Normalize OCR text while preserving sentence structure for SBERT."""
    text = text.translate(_OCR_TRANSLATION_TABLE).lower()
    text = text.encode("ascii", errors="ignore").decode()
    # Keep punctuation that helps sentence encoders, remove OCR garbage
    text = re.sub(r"[^a-z0-9\s\.,;:!?\-/'()%]", " ", text)
    # Collapse repeated punctuation and whitespace noise
    text = re.sub(r"([.,;:!?])\1+", r"\1", text)
    text = re.sub(r"-{2,}", "-", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Demo
raw_samples = [
    "Machine learning\u2019s key advantage is its \u201cability to generalise\u201d from data.",
    "The syst3m us@es n3ural n#tworks!!! to pr0cess inp@t.",
    "Non-ASCII: caf\u00e9, na\u00efve, r\u00e9sum\u00e9",
]

print("=== Step 1: Sanitize ===")
for sample in raw_samples:
    sanitized = _sanitize(sample)
    print(f"\nRAW : {sample}")
    print(f"OUT : {sanitized}")

## Step 2 — Fix OCR Split Words

OCR engines often break compound words across a space: `applica tion`, `integra tion`.  
This step merges them back using:
1. Known domain terms dictionary (exact match on merged form)
2. Suffix heuristic: merged form ≥ 8 chars AND right token is a known suffix

In [ ]:
# Source: backend/core/nlp.py — _fix_split_words

def _should_merge_split(left: str, right: str) -> bool:
    if not (left.isalpha() and right.isalpha()):
        return False
    if len(left) < 3 or len(right) < 2:
        return False
    merged = f"{left}{right}"
    if merged in _all_domain_terms():
        return True
    if len(merged) >= 8 and right in _SPLIT_SUFFIXES:
        return True
    return False


def _fix_split_words(text: str) -> str:
    """Merge OCR-split words while preserving surrounding punctuation."""
    parts = _TOKEN_OR_SEPARATOR_PATTERN.findall(text)
    if not parts:
        return text
    merged_parts: list[str] = []
    index = 0
    while index < len(parts):
        current = parts[index]
        if (
            index + 2 < len(parts)
            and current.isalpha()
            and parts[index + 1].isspace()
            and parts[index + 2].isalpha()
            and _should_merge_split(current, parts[index + 2])
        ):
            merged_parts.append(f"{current}{parts[index + 2]}")
            index += 3
            continue
        merged_parts.append(current)
        index += 1
    return "".join(merged_parts)


# Demo
ocr_split_samples = [
    "the applica tion enables integra tion with the redis service",
    "yaml configura tion is used for kubernetes deploy ment",
    "reinforce ment learning is a type of machine learning",
]

print("=== Step 2: Fix Split Words ===")
for s in ocr_split_samples:
    fixed = _fix_split_words(s)
    print(f"\nBEFORE: {s}")
    print(f"AFTER : {fixed}")

## Step 3 — Domain Detection + Fuzzy Correction

Detect the subject domain from hint keywords, then correct OCR misreads in domain vocabulary using RapidFuzz.  
**Safety**: only applies when fuzzy score ≥ 80 AND the margin over the second-best candidate ≥ 5 points.

In [ ]:
# Source: backend/core/nlp.py — _detect_domain, _fuzzy_match, _domain_correct

@lru_cache(maxsize=512)
def _detect_domain(text: str) -> str | None:
    """Infer text domain using lightweight exact-match heuristics."""
    tokens = set(_TOKEN_PATTERN.findall(text.lower()))
    for domain, hints in DOMAIN_HINTS.items():
        if hints.issubset(tokens):
            return domain
    return None


@lru_cache(maxsize=8192)
def _fuzzy_match(word: str, vocab: tuple[str, ...]) -> str:
    """Return a safe domain correction using top RapidFuzz candidates."""
    if (
        len(word) < _MIN_FUZZY_WORD_LENGTH
        or not word.isalpha()
        or word in _PROTECTED_COMMON_WORDS
        or word in vocab
    ):
        return word
    matches = process.extract(word, vocab, scorer=fuzz.ratio, limit=3)
    if not matches:
        return word
    best_match, best_score, _ = matches[0]
    second_score = matches[1][1] if len(matches) > 1 else 0
    if best_score < _FUZZY_SCORE_THRESHOLD:
        return word
    if best_score - second_score < _FUZZY_MARGIN_THRESHOLD:
        return word
    return best_match


def _domain_correct(text: str, domain: str | None) -> str:
    """Correct OCR noise only against the detected domain vocabulary."""
    if not domain:
        return text
    vocab = DOMAIN_VOCAB.get(domain)
    if not vocab:
        return text
    def replace(match: re.Match) -> str:
        return _fuzzy_match(match.group(0), vocab)
    return _TOKEN_PATTERN.sub(replace, text)


# Demo
ocr_noise_samples = [
    "rubernates deployment uses a helm chart and yaml templte",
    "the kubernets servise uses postgresq1 and redis configuraton",
    "helm chrt values are stored in yaml configurationn",
]

print("=== Step 3: Domain Correction ===")
for s in ocr_noise_samples:
    domain = _detect_domain(s)
    corrected = _domain_correct(s, domain)
    print(f"\nRAW    : {s}")
    print(f"DOMAIN : {domain}")
    print(f"FIXED  : {corrected}")

## Step 4 — Tokenize, Remove Stopwords, Lemmatize

Only used in the **TF-IDF path** (bag-of-words). The SBERT path stops after step 3 to preserve sentence context.

In [ ]:
# Source: backend/core/nlp.py — tokenize, stopwords, lemmatize

def _tokenize(text: str) -> list[str]:
    return re.findall(r"\b\w+\b", text.lower())


def _remove_stopwords(tokens: list[str]) -> list[str]:
    stop = _stop_words()
    return [t for t in tokens if t.isalpha() and t not in stop]


@lru_cache(maxsize=1)
def _get_spacy_nlp():
    import spacy
    try:
        return spacy.load("en_core_web_sm")
    except OSError as exc:
        raise OSError("Run: python -m spacy download en_core_web_sm") from exc


def _lemmatize(tokens: list[str]) -> list[str]:
    if not tokens:
        return []
    protected = _all_domain_terms()
    nlp = _get_spacy_nlp()
    doc = nlp(" ".join(tokens))
    lemmas: list[str] = []
    for token in doc:
        if not token.text.strip():
            continue
        if token.text in protected:
            lemmas.append(token.text)
        elif token.lemma_.strip():
            lemmas.append(token.lemma_)
    return lemmas


# Demo
sample = "The neural networks are learning from labelled training datasets and improving their predictions."
tokens = _tokenize(sample)
filtered = _remove_stopwords(tokens)
lemmas = _lemmatize(filtered)

print("=== Steps 4–6: Tokenize → Stopwords → Lemmatize ===")
print(f"\nInput   : {sample}")
print(f"Tokens  : {tokens}")
print(f"Filtered: {filtered}")
print(f"Lemmas  : {lemmas}")
print(f"TF-IDF  : {' '.join(lemmas)}")

## Public API: Full Pipelines

- `preprocess_for_sbert(text)` → clean sentence (stops at step 3)
- `preprocess_for_tfidf(text)` → bag-of-lemmas (all 6 steps)
- `preprocess_markdown_for_sbert(text)` → for Mistral markdown output (no lowercasing/stripping)

In [ ]:
# Source: backend/core/nlp.py — preprocess_for_sbert, preprocess_for_tfidf

def preprocess_for_sbert(text: str) -> str:
    """SBERT path: sanitize → fix_split_words → domain_correct."""
    sanitized = _sanitize(text)
    split_fixed = _fix_split_words(sanitized)
    domain = _detect_domain(split_fixed)
    return _domain_correct(split_fixed, domain)


def preprocess_for_tfidf(text: str) -> str:
    """TF-IDF path: sanitize → fix_split_words → domain_correct → tokenize
    → remove_stopwords → lemmatize."""
    sanitized = _sanitize(text)
    split_fixed = _fix_split_words(sanitized)
    domain = _detect_domain(split_fixed)
    corrected = _domain_correct(split_fixed, domain)
    tokens = _tokenize(corrected)
    filtered = _remove_stopwords(tokens)
    lemmas = _lemmatize(filtered)
    return " ".join(lemmas)


# Markdown-aware for Mistral output
_MD_HEADING_RE = re.compile(r"^#{1,6}\s+")
_MD_BOLD_RE = re.compile(r"\*\*(.+?)\*\*")
_MD_ITALIC_RE = re.compile(r"\*(.+?)\*")
_MD_LEADING_BULLET_RE = re.compile(r"^\s*[-*+]\s+")
_MD_TABLE_SEP_RE = re.compile(r"^\s*\|?\s*-{2,}[\s|:-]*$")
_LATEX_DISPLAY_RE = re.compile(r"\$\$(.+?)\$\$", re.DOTALL)
_LATEX_INLINE_RE = re.compile(r"\$(.+?)\$")

def preprocess_markdown_for_sbert(text: str) -> str:
    """Markdown-aware: strip syntax, preserve case + math for SBERT."""
    lines: list[str] = []
    for line in text.split("\n"):
        stripped = line.strip()
        if _MD_TABLE_SEP_RE.match(stripped):
            continue
        stripped = _MD_HEADING_RE.sub("", stripped)
        stripped = _MD_BOLD_RE.sub(r"\1", stripped)
        stripped = _MD_ITALIC_RE.sub(r"\1", stripped)
        stripped = _MD_LEADING_BULLET_RE.sub("", stripped)
        stripped = _LATEX_DISPLAY_RE.sub(r" \1 ", stripped)
        stripped = _LATEX_INLINE_RE.sub(r" \1 ", stripped)
        stripped = re.sub(r"\s+", " ", stripped).strip()
        if stripped:
            lines.append(stripped)
    return " ".join(lines)


print('✓ All preprocessing functions defined')

In [ ]:
# Side-by-side comparison: SBERT vs TF-IDF preprocessing

SAMPLE_TEXTS = [
    # OCR noise + split words
    """Rubernates deployment uses a hem chart and yaml template.
The applica tion enables integra tion with postresql service and redis.""",

    # Normal student answer
    """Machine learning algorithms learn patterns from training data.
Supervised learning uses labelled examples to train classification models.""",

    # Mistral markdown output
    """## Key Concepts\n**Neural networks** are computational models inspired by biological neurons.\n- Deep learning uses multiple layers\n- Loss function $L = \\frac{1}{N}\\sum (y - \\hat{y})^2$""",
]

labels = ["OCR noise text", "Clean student answer", "Mistral markdown"]
processors = [
    ("SBERT path", preprocess_for_sbert),
    ("TF-IDF path", preprocess_for_tfidf),
    ("Markdown SBERT", preprocess_markdown_for_sbert),
]

for label, text in zip(labels, SAMPLE_TEXTS):
    print(f"\n{'='*65}")
    print(f"INPUT ({label}):")
    print(f"  {text[:120].strip()}")
    print()
    for name, fn in processors:
        out = fn(text)
        print(f"  {name:<16}: {out[:110]}")

## Sentence Splitting

Used by the sentence-similarity scorer to align student sentences with model sentences.

In [ ]:
# Source: backend/core/nlp.py — split_sentences

_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
_MIN_SENTENCE_TOKENS = 3

def split_sentences(text: str) -> list[str]:
    """Split text into sentences, preferring spaCy with regex fallback.
    Drops sentences with fewer than 3 tokens (OCR noise filter).
    """
    if not text or not text.strip():
        return []
    sentences: list[str] = []
    try:
        nlp = _get_spacy_nlp()
        doc = nlp(text)
        sentences = [sent.text.strip() for sent in doc.sents]
    except Exception:
        sentences = _SENTENCE_SPLIT_RE.split(text.strip())
    return [
        s for s in sentences
        if s.strip() and len(s.split()) >= _MIN_SENTENCE_TOKENS
    ]


# Demo
paragraph = """
Machine learning is a subset of artificial intelligence. It enables computers to learn
from data without being explicitly programmed. Supervised learning, unsupervised learning,
and reinforcement learning are the three main categories. Deep learning is a subset of
machine learning that uses neural networks with many layers.
""".strip()

sentences = split_sentences(paragraph)
print(f"Input ({len(paragraph)} chars):")
print(f"  {paragraph[:120]}...")
print(f"\nSplit into {len(sentences)} sentences:")
for i, s in enumerate(sentences, 1):
    print(f"  {i}. {s}")

## Pipeline Visualisation

Step-by-step trace for a noisy OCR input.

In [ ]:
import matplotlib.pyplot as plt
import textwrap

def trace_pipeline(raw_text: str) -> None:
    """Run and display each step of both preprocessing paths."""
    sanitized   = _sanitize(raw_text)
    split_fixed = _fix_split_words(sanitized)
    domain      = _detect_domain(split_fixed)
    corrected   = _domain_correct(split_fixed, domain)
    tokens      = _tokenize(corrected)
    filtered    = _remove_stopwords(tokens)
    lemmas      = _lemmatize(filtered)
    tfidf_out   = " ".join(lemmas)

    steps = [
        ("Raw input",           raw_text,                    '#e74c3c'),
        ("1. Sanitize",         sanitized,                   '#e67e22'),
        ("2. Fix split words",  split_fixed,                 '#f1c40f'),
        (f"3. Domain correct ({domain or 'none'})", corrected, '#2ecc71'),
        ("SBERT output ▶",     corrected,                   '#27ae60'),
        ("4. Tokenize",         str(tokens[:12]) + '...',    '#3498db'),
        ("5. Remove stopwords", str(filtered[:10]) + '...',  '#2980b9'),
        ("6. Lemmatize",        str(lemmas[:10]) + '...',    '#8e44ad'),
        ("TF-IDF output ▶",    tfidf_out,                   '#6c3483'),
    ]

    fig, ax = plt.subplots(figsize=(14, len(steps) * 1.2 + 1))
    ax.axis('off')
    ax.set_facecolor('#1a1a2e')
    fig.patch.set_facecolor('#1a1a2e')

    for i, (label, value, color) in enumerate(steps):
        y = 1 - (i + 0.5) / len(steps)
        sep = "▶" in label
        bg = '#0f3460' if sep else '#16213e'
        bx = plt.Rectangle((0, y - 0.04), 1, 0.08, transform=ax.transAxes,
                            facecolor=bg, edgecolor=color, linewidth=1.5 if sep else 0.5)
        ax.add_patch(bx)
        ax.text(0.01, y, label, transform=ax.transAxes, va='center',
                ha='left', fontsize=9, fontweight='bold', color=color, family='monospace')
        wrapped = textwrap.shorten(str(value), width=90, placeholder='...')
        ax.text(0.30, y, wrapped, transform=ax.transAxes, va='center',
                ha='left', fontsize=8.5, color='#ecf0f1', family='monospace')

    plt.title('NLP Preprocessing Pipeline Trace', color='white', fontsize=13, pad=12)
    plt.tight_layout()
    plt.show()


trace_pipeline(
    "Rubernates deployemnt uses a helm chart and yaml templte. "
    "The applica tion enables integra tion with postresq1 service."
)

## Interactive Test

Paste any OCR text below and see the preprocessing output.

In [ ]:
# ── Edit the text below to test your own OCR output ──
your_text = """
The reinforce ment learning algoritm uses a reward signal to train an agent.
The agent interacts with an environm ent and learns an optim al policy over time.
Q-learning and policy gradient meth ods are common approches.
""".strip()

print("INPUT:")
print(f"  {your_text}")
print()
print(f"SBERT  : {preprocess_for_sbert(your_text)}")
print()
print(f"TF-IDF : {preprocess_for_tfidf(your_text)}")
print()
print(f"Sentences ({len(split_sentences(preprocess_for_sbert(your_text)))}):'")
for s in split_sentences(preprocess_for_sbert(your_text)):
    print(f"  • {s}")